# ASC TCN five-seed T4 rerun with trajectory latency

This notebook reruns the manuscript TCN baseline using seeds **42, 52, 62, 72, and 82** on an
NVIDIA Tesla T4. It preserves the original nine descriptors, sequence length 20, recursive test
protocol, optimizer, early stopping, and 119,234-parameter architecture.

The latency column is a synchronized end-to-end recursive measurement in milliseconds per
evaluable test-cell trajectory. It includes input scaling, host-to-device transfer, batch-one TCN
forward passes, device-to-host transfer, inverse scaling, clipping, and autoregressive feedback.
Each seed uses five warm-up rollouts and 100 timed repetitions.

No Kaggle dataset needs to be attached: the three processed input CSVs are embedded. In Kaggle,
select **Settings > Accelerator > GPU T4 x2** (the code uses GPU 0) and run all cells. Download
`/kaggle/working/asc_tcn_t4_latency_results.zip` when finished.


In [ ]:
import platform, torch
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before running this notebook.")
gpu = torch.cuda.get_device_name(0)
print("GPU:", gpu)
if "T4" not in gpu.upper():
    raise RuntimeError(f"Select a Kaggle Tesla T4 accelerator; detected {gpu}")

SMOKE = False  # Keep False for the publishable five-seed run.
LATENCY_REPEATS = 100


In [ ]:
from pathlib import Path
import base64, gzip, json

ROOT = Path("/kaggle/working/asc_tcn_t4_suite")
ROOT.mkdir(parents=True, exist_ok=True)
packed = json.loads('{"run_tcn_t4_latency.py":"H4sIAAAAAAAC/7Vce3PcRnL/fz8FDlXJATIIk7Qk25vbq8gW7XPFUnQU7aRqs4UCgVkSRyywxoPimmE+e349PS9gsRRl+1imCMxMd8/09PT0C/Z9/7viVhy1QuTexbdvvUY0feV9KLprr91V2XVTV8Wv1Pfc65r0HyLr6mZ3VKadqLKdtxFp2zdiI6ou9n1/Nls39cZLknXfoTlJvGKzrZvOS6uq7tKuqKt2NtNtzdU2bVqh3//R1pV+3oLAum42+r1Jq7w2b+113xWlfuuKjWC6edqlWZm2rWgN4TYvsi6yXTxym3bXZXGpR73Dq5lX1W+2OwB61dZMB+TRgP+2uSFbN9n14CWuKglWMY32phRpU8Ub0TVFZmbUnCZtVjdiOGjbiG1TZ6Jti+pKD33fEd0mf5+lpWgYgCnR+tuYFqXHvsbzj3WaiybyLkTV1g21tKKbzWYXr86/P7t47y28pf93P/L8c+GvZt+dvbr46fxMNs88/Pg3ibjbUn8nNlvRpLSH9JoleBZJdn3lvOVFqxraOks+FNigD37EiNIrkXS7rQT++zFT5D9Zh7+z1ez92dlrSfn5aeS9wO9L/H6J369OV7Oz/3539u3F2evk3avzV2/OLs7OaejJydfJ6RfPZ6/Pfv7h2zM0MCtycVtkIvCzPk/9EKv9d7PXATj2q6gWF00vwpls8r6tq3VxNZfzbMUvSSmquVdUHfCdHsvW6yLPbePXL2Vj2cy9dVmn1PSlOHouGzfpXSK2dXbd6tEnXzEOyFeBAyJ0+xcvZPNl2mXXSYsDpTtenJzKng+iuLruklxk6c5SOiFKs1ku1u5hDELv6K/e27oSvAzmA60/HozSkKJL6HgH9I+kO4LnwxWbMaFsrbbxdAeT26RVn5bJZJ+cijMgSctyb9Blmt2IKm9pdFXFl2DX9SZtbrDs79ISauHg0Fx0otkUVdF2RYbhtL1qrZt0K4Jd0qFlTiug49Oku8jbJThhudsmmSAZzVzAntdAmoIkcGJcVhbbAH/Ty1b+aSWYwg6NQgK+kAjCMKKt+iqSTOVFNgKnp2ICBA5NWWlsCod3pKYVep+75MPQe+adHB/Hx3oLlQr5lJWR2lu2HbSBnMJKbbVo+7Kb7/VixfcPzHGoZQgMMbKl87mSrdDFEJxc3EVqAN48AU1JWkIESsGETIR+btMSC1y4G7KcR4xjpSdtW0ID6NKP0+0W+x5IZHYIL2K59u958EPy5tW7M58WIUc+MvD8zXseyPtiRiqBb39p7GbtTdvslzNv79kz7zS0c3tsluenDml9DTyFOaGzd0t/k2ZNbZc8lDCXfeFAEhlcCRRg8qTdlkUXFNW2h+IpoODoGpTC0/XbUiy3eUx3yHdNuoG8H35TsrWmZ0wIXY0A+qy9tcgh4f72GvfRSZLdkj5ImvoDDnR76/MssXNFLi2EJBNlSbIHxWV36CDWdXEn1FoSIJG8g3STqcHolz4hTIrcX8V9VfzSk2q0e9WJtvsdJCX4J9NsUqgvXPQL5try//iPA1XAEAj2mPK/znTDVZzV210w5p9B+jScQzSE/iMIDs2gkFdNoNcWK7DQ+1fZbumaHqsuANMK77yvyJg7a5q6CfwLhehzZ2kE6cFiuoGF4T+JLM32UwlKJhwmNbGSTyf2s0FykJw6uHplkbPHkdwodZgv+6KEOApIGUyONrCncT46tWshbTqYKyUuT7oAoG40IJlCV921tBBmjhpwLxv7vDImFANnddlvKjpDmoj3mafuhcHIVl8hfL1E7hWTRN5VU/fbpCcDtoPbgWtGriSW7Ze7wEgjJo4hC2krOLyW44B5iCemP4m8HdpA2bkhTjfZRlLJBnlTb5WhqFHJdWpUy/FKV3FXJ9JXIOUrtfAXp3v32BCH4sdHQM1lS6uHDXYlgtEeRZCUKpA4Q2ftAzbry1POYMn4jvY2e3z92qkbBO5Klu54HAd4dZaiI/AsuXv3K9yKbhcEx3tCxwsas5jsKodB0WF0BK2NkENAg0NFlz28hBtDs2UwblQcIPOLvYZz0RY5zFl4yN+UNUbADH1T532pRY/OYQJRwo2QAGe5jrzsGl6vKNk3gL1YlPLo6lfIW9131tY/jk+crWx7OGBBGBucdosIe7xN85wvkFMYixr3cExWV7dgpLoTMGP4Prcn+XBX9Cwj5+lGNHiQfsriCzvzhX7A9c/kF+5cJmwg2V3Bj4fi+jW1E/mG/KC3aMdsNNkRlOIPj3/NL4FqxL5onuOsfICHrFgub2kwnN0GdoNZkzkNlslARePlvT/kF9/3zpyk6rdrHZ45i0c9ke1G/x25MKtDdNWqAp5kI8o+2GdcoIDCMBwfMweOpw3Nq0cbCYbkvoH9VzxFcrWxg7BOayV27Bf/bhlmMgh7UFRpLKSjOWjyQ+E8GWG8pLPZMp738mB3RVoOBf7Zcu8wa9RWvrWgSV2sm0kdByeRhyjFcwQqwtUhib+G1fj4JND1Y1Eh8mNov3xOCqiKz8WPPwX8OJb6yAF7iRmcum7HH3MieDZaMJmdweRmKVGLYZ1U7bZuheTMhHQajqilqrNxdLLS3q00cBJ4zhxNGFjIIwtmZOpO9ZooB79mHO9RcZ/INWz0kYhGgbb995EV5TjUg+AKr/0ukZMnP04+gJtjC82adDoIF6mJxiompVG5Jt8uGRj5Y6Tu0MfRqoubLk012dCDnNs2i+ojVuzb2voywODMz8zLV7LZSl4miCCoJzKLhowmuR+3sG3IJiXsgl4qHzXruL2Gn7uEKLmMJ+lVtOI1eWgkoRRM1oslu48Ag6OTaIg6DE2fWfaAFvtwiAVR8GfPeLsbbpCZhZ2Bg9XxIg7PR9HTAwZQsungXHZjZuxGzFADHkEwtZhd7GJw5GQKDQdJ61yUFKXTF9BghVqvL5Sc8lsIAzngOG+o4ql09BD2MyLQ9pvAtMYUiSoDVtemlfS1pB6bpjYIzREY4/zTwpsIPD8q/wOdvvYpf2KpM9b7EZl59ECU4IBAj8LBuZ+iGT34o7ul3oIqrrzGxL1lS/wqTzf/Zeexv1zYxo1mbglV5gaadbvb5kQpEN8XZCiMaZZNYrpwX+V9Jn48/8/qHSWF0t5OxswZUp1S3mhxHL+ITGx8cQLDHSHPBBNE9PSFQ/lKVBRYrC3l73UL7Inp0HMpkx8AsJkQO5VBRkQZWp1sMxow8gbN+ng4DoSN3mvG2RY7Csmp9boU0pe0rWZFC/MUOQt2gjJ3Zs1mgq5m53THwj0cDvBuDLx7DJhzEhTOKesWdr98hAfU4ZK7ROCLcxscDS+qNYe3I0+nSVpKBdoY8TD3IFvgR5GML2SOLoY1uOZDYBU7tlhSsa7uibmzbHoFBq1rWLKUy/1xbEpagxuz1vh5k+4i9bAjUiwsQytejaOblZ+sEoLlBeZKc4i8nmGYwMLuDOzuE2DNMYnxWydXTUpiDVWNGAH4PQFBK2X7ct1X0hpLy3jTCrmNgeROoNYQmmXvo5B5FWkrhmPjlPOLlAGR80nIFUmCKe1yQlmK6eW0ndgG+2RtaIJD13IqBVKOgWs9Mi0B6XVQyLQ0yzfkUTQyWEAjg1EUxDkSilkTAX+11sM8dE9lGA3OWahmPEDouAFGP0omjKYTOva2PESaIfcDdL4Ufn/OJ2QY/fDZaCZs/nyUA2Amh6N4iT+aBMBGLaPxMjENkZXpXkPEbq+Ug0QGodrl8Wrpl42/cog+DLzn8Y78xSoehKWg/78cnUbTuxjD7o+TOotyWHv7eyN2c84KUdowza5xf2Tbnv4tcbhGG6hVBqAilcIyNoSkkVCYP+DNb0fADyON4OjPY9OFQIeYHxr4GTK+Ls+crr8utFY0qeUhFmQqbg6oYW2h48YEjryd1scUHGR9rQ0kh7VF6+SLPxY/9ypxi5sYviKZBrmXIrQkspttDadMhbaZpZyFsny1BAexOjk4mnAgho5ZoA4T3eSjFStHsxFZ37Qod0HyqSzJtbaTmRsDNRq4K/OxM+j27qZ7p2Lvw2i5E2rXbmpZwiCce5d1XUYfD79PZuD2dOZEdN5UfQyj88hBdtcwAZxMVkRp51TWmOhAvQnWP0kR00lS0frfH9f/g2P7SsRtGN37y2Jve/Z0A4SpKyonwyyPHuw9aXh+Sp7ATZz9IXF/3hUqwpGhAZrSR4L+e+BSqPNpx5VRT7jH7s/JU+P6e9AHvVD3x8omFRbIG5pFkAqjFIt5Da7/qPU9938Eq/FyiwoqDNaA5YAdF+K2exyNrh6xjTg5uONeyjqRr1duoci+KC1VpQWxTh/XcD54w+k9peS/JbCHC9KtlcrkZvGJ15aHlCko5EwTfzRVFU5idLPgCq3DtEkQrWGMAWTUwNyqDnWm597eJFXP6mEQ19pbts352IzRqcoTjVoGF8o4Y6RLgJh1DK/bnMWP0ehF6ninKXVKbEVloioq/0k3EmWLP+1CcqbGcQvbhVJFhHBapwVOxAZWoClqG1xgUmhU1DTixDNHUNWthbrRN1xGOqw4hTwcdfUR/tiL21MXt6fYFbMPewGTlMomq6zsc/iAxAwZXr2u246QsOPr8WmmgIR0io5wCGXRq4qZRyqITmMJiqAdGKUTLHY641v5BI4jcidy8qhiT8UZpJdJ3U4cTDdJCC7r8oAu61WKIQUXxJ1cRh5r9tj8uL0ZFMvd+OyedfOI4UQCMaGt1dFRt+8htz7dYF9H1WHO1JR8uJmgPRxPCw/8E5f1yLR4eVolBY/ayqpUD7V6KAwan5lHKgEVEVI5pqPtct2OUE1er5HfQr9q0jZsXyGlf9VTyTUOnC6ijt/iTLfbNBPy5FEl19zViU5hZtEm6W2KnNZlOTDXpkz6s4pGwYL/j/TqCg/fv/vJSzNoZhWeuxTYe0GTkoZ/BxOcyAlYsTfK1L/a9knFlWHOJKiigA+a7AyOjQb3L577csqQJw0b91uZSnx0soi/EnkgzWnzaVLg/S99QbUoaeW9/fmH1z+8QiH7v3lUQcrRV03hQacpOHGKyRomx9xExV9k/tTlrZYW7og3N+iiYDSN5rgfzjD0XFLfOGbn4ToeUHMK8ixhU3Rm6YZOUgtgnNVy4r8mZLY4JWZaXO2mvhHS/aQiZeulm2js4fFfvHBDlRRxpbMvK7iX85PVQTg5gmOzrK0TpRmoIPoglG0cQfFhQtVgpMImhRg4LIl0KNCk4Vw9FbkejFFa8quHouI1jeKL0wpG+ZjRpFM9kce01tbU5tM4He6cSCbrUmMuCB34Ywdc2SesgFXkMC1oNaSspd6fyVizYQa46o1l49YIVrre0YnMqg0hzedsT5dHzl3ycavoty5uvBGDlURj4YwO1vLuBZh82j/YpXIbhz1yjuiinBC+hinLr/VG+cORz54Ni7v1bo+jdzbqCqwTOaPJ8KCSTUCMxXU03HHi2wSq1tkDAA/278mAiqG0zw4O2vVRnHHIf2fs3o7I8VVCEQ5ZP0xj4RYx58LpcbJSlTkw3PiJWKtGaCJJwzF8YZEjom4NJ9hpZYXqmpXtwHKzF+5dsuCsZJ2FirfZ7sLaHnoes7G/dAiD6ceHPpIp/O0NP65oOPNqGoDkTgHIRwVAz7N9J0/pWz1X6+WMj702qMRdRwMHPL0fnCHP5y335yoQ8hnFFPwpmZTxnofp+IkTu7XfKGiLa+J4bxtSZqNkLk1ocU//Pni25H5xr8rw/2zb/ryaxy/XD//i+SMUigGLe/cAybHepv3cLodP5+J+fEjn8en6ofXHVT5N+oHL7K2LSUIX6k5y2almXZkyKFjvsirpnvO3OErdyDp19YWBaxcDLWhnsEidS7a4QgJIcDSNLRpNY1TT5hAzi9E5jj2CjlFhqU7c57+F+rgi/xHClosjgX0CmZGmepQUKhZwve2cWOxyZu8L+1EHfTNnnnAU9eNQpVvV86jyftowV1UzxEoL03I07VUMTyDAVygQZfnxH0BW4WGRY2j1oYf6aq4q1mz12uvUp4KIhqr96PDjCwv3m1R8e+p8lqp47i5tu+uu6wqA+rvRmFsSctipWM3R5L70QnxdEpfoMUnijCEfJWnYuzBD1UDpwAzHVpUZNPpWbWoCuD0m7xCpclqlC1uXgrRn+kZajejnj1oDbnYx69J7jDEVYM7CuaoZnSq453LwyebFp9zXZqyKVmDsi4leWIxy1zdFifCQsqchDh5d4L30QMmwO5LfSEwLtzm6iKvTjgUjQ+2pgaW5je787fS1GycaKndfh4y816d/+2hoKO0xX3HV0Oe9oKUjRY5md7eJsmDKuqF7iQwSagv3BmnThka5n544NrgDYpiZjCnQneFYVXFBIdbj1SPALmUHWjZPgUv/zufsVjBy+9Q4vsv3NYiobgvsHAEkWnXE9K04kjwfGiRiMaM75/amrjhHxBpZfDWcNXLVLU4pcO7/DykuSEstS9j9vlsffeXPnAs2xUGmfXKDAKptFAHQrez8PzUSoKASUmdYL5TkGp+USXKMjjpUa+CrOEqGDC9Hyyi4E/A38KiRuRGJAgvwPU8wjRthJP/Xgr7tZv7aKjiN9k+GuvWDVR/WDL2Kb6zVAHVzSqNpjRuK7KHWQHv36unBC/SjTJ4jB0PpXRxB7O6JeCmtG+/NN+HwcycFosJdMriFBV61/NXzfrxrrgsEW5mEMwNeqa17J3uC0BkWoyA/0Vsb+EdHMtByhL0Dg+RnvcTiSEePcmfrDiBgrv4eDGrZvxVc6b8jrZQVGv7URKxTbBHq7Y4fxcGHFBpL2k0Ln8w25ToMdkiBu1uD3SroMwK60PB/fVgsPD/BaYW9kvjqy2PELV2IcPb/2wDZXXtCAAA=","inputs/phase1_cv_all_rows.csv":"...","inputs/fixed_split_val_predictions.csv":"H4sIAAAAAAAC/62Z3W4cNwyF7/MsgwUpkiJ1HfSiN0F+HmBgONs2iNMa9qbo4/dI45/VTDGGdgvbsC3b+w0l8pBHvj3e3c3fvk7f5+M/99Pp+OP++HBz+vlwnG5nfHGcb//4fXo8He/nb3/Ot/jl6cdfX4930+P93bfT9Gk+Pfw8Tp+Py+dP8/3D8Wv9tn5+95FI5jR/oTS/Z57oQC8fiaaPv3748Nvx5jQ/3t88PB7n7/PnI82fb080f6L5l5svp4fj4+P0983dlA5OFjGxHziULNUVNrKsdS0JkfkujkdxJFZ6HkVmlYWHn/MuL43xctFYXvuMlwp5bjxlyrHLk0FeFHHteLl4LtkXnoWmXZ6O8rKzr3jJ8d54xiT7PBvkeelPL0dQlCVbkDW0ny15lGaJ1jyxFEt0mYvu4nwQl7OmFc4LBS/hZQ+yXV6M8nRzeI6laM+QXKKF5zMDl19wfGmt1zT0xiuFk3orhuwpQwDiQKgLLftAHgUSL0pyBgQx8gIsto8bLfYwTtHhcrEk9fwqDqUS+8DRaneUe14BKUU8baiGpX3gaLm7hq8iDMvGtgDF3+CNljvEJOmKx6xP5yep5H1eHq6ILKv4HG/pGchvJOhoxRuOyFa8JPk5Y94OcLTkIV6xAuYgTs9Aj3qCnCqQXyUm2aU1L9ARNMB8cFcRrUtqLIKHsEMJdMA3gKM1zwis5wlKMOfGU2Ip+7zRoqeckNYdMJGDWYHhlluHZ62izQHgwtGLVTQVx0PiCB1qo9x2VIKf1tDjdZ83vqFLQ3jFIUGlVaULlSz7uNH9RCDLBPPKQxWYLzxOTbR3ePI/8CTQBxeg5PQGcFRDyShW51f7ryxr4sX3eYMaasWYbcUTDGZL0CpQ733goIhaYOD0HkhelPMCLMV5H+ijQCFdAzFVLDyDgL/BGxRRc7xYz7PiqkuVZBxwrXiJNqb5/J58kkT5VbchiGNVz4eClMz4M6AMMgNxOQTGT/iHulYSRM42UCHpoTwGhVWBWTpnQmmQTKmuIWVzldLMsyz7ilE5ccjr1g7HmQ6aLLV2kTRzbU8V3BZYE0aoDVDaxuqFMeL10fvwN+fEpAIBb0xTCOqGqaI9c1Dj0IWSp47JjmnUGxPyWrZxWqaeOahzUiSp9kzKzSyC2RJ5w8Rs1zMHpU6C2ihzxiTFjBULM5LZhonxrmcOyh0sPCT7nCklgqUtMYbV5BsmiqtnDioegFR75Dnzebe5VCO1RiJrV2k7qHliskwdL8SANV4C51IwX22Q6J09clD2BESSninUzjLBPbYY43lQrb/IEvkav8iYqNBIEmHi4KqBJVxKagtVkdKWiKO+xjCi9pj0jEh1W9uCkyOLNkQIpF7jGVnV4PlfiGgfuYUMk1p4gxOY82scY71MUHvF5XpXQ1WmMYuX0kbHFRGW/XLLiC6l6FNnh6gYz92rTkNdcWDbQ1SU0+WmkQ+m9UbhBQiJQ/1bA7py1E31pTDkKWvwTNd4HIUhreVvGDsK1y6NJVwU1QnLMC1I4SZ0PTUh064xOkrYx9xTCeJnCxUTrG6ZeJZrzI7gQqVeLpwxpXjYEr2VkFYiPRRalDroaOfCAKfWQwP3AuVpzahsIxV00w462roybkrXULRobbYuM+NicANVy/32jvYubKSutteTVz9bmbn8R/IaR7+7o70LEkt98kr1k8RtDd5WfQv10ueuD7eSnFMPNYwKqgsUmrONFKaiP9LR/oWMSdxDNQBZigg/9trDPLerAQNUJ7ijLNe4da59Cg9fDolxFZBbk8Hwt1zRBcbLsmHCz8Q1jh2qalx6JkRel2uzQBG3GaiHBp8Pe+PSQPAnNWfOoOS4ba1OsEJhoze7a7ihzNeYdzK0kOihUsulEt23YRq8vl/j3oko4nxvMSQgh55uQEO0memeaZT0cgePFhrYPO+ZWEA2N2aCGG2YyDC/3MRzvRfALVbHxL96MNBxZXqpOfbuX4m8Y/xkHAAA","inputs/fixed_split_test_predictions.csv":"H4sIAAAAAAAC/6Wa3W4cNwyF7/MsC0P8E6nroBe9CfLzAAPD2bZBndawN0Afv0fS2l7NAGPIiyBxIif7DSnx8IiTu+P9/fLj++Hv5fjfw+F0/PlwfLw9/Xo8Hu4W/Oa43P315+HpdHxYfvyz3OEvH37++/14f3h6uP9xOnxZTo+/joevx/71y/LwePxe/1i/fvicki68fMOvH4kO6Sa9/OR0+Pz7p09/HG9Py9PD7ePTcfl7+XpMy9e7U1q+pOW322+nx+PTEx7p6XTgGycpqRwo43deWNuSRxSrazmnkm0XSLPAlDOteGScG86Sxz6Op3FkriMv2fkRslBS3uXJJC+XrLHmpZTxQRVIyUx3gToNZHIfgLmoKLeHsPAcsgu0WSA+kngERohQS6llCtlPaZ4GiuU1kEKpnRkTljfOjM8C3cxWQLfUy0RLyZJ3eTHNE+fxzOQcyaIFqJlxpCrQG7AA2EH07qrPxb2WnN8oNsylF0pRt7aGA5T3gTQNpExlAALikaKtQXD2eTx9ZqwIj7wIyqZtLSTRPnC67iO5xAqIjJp0ILbzjZTq/Cn1IiPRrUT0sIt46D5xuvKhnEIrIo5NtKSiQPiNpE5Xfjbz1S5mNSPvQKfM+8Tp0jdPK6CVFNJqxUjprV2crn2zJKtdNPF+bIzJUo2QaKHlG6UXrWF7d+k7+lML0Yqnwm2pJBymupZTCpN94nTt51BfE8nOBxUOo/f8HeJ09UMyI0Zi1nxey0SR8z5R3nFurIxEC8qdp6j+fd508Rux0oqHSkxnIvrxPnC69tUKr1KqGRXfgcziuk+cLn5N4boiJjTIDvQkZR84XfuCU7PaQ1GoTHsIfDf5PnC69jlyrAqDS+IzD/2kpdQWAdBfgPrutq/4xO4VDf6eoS/+7FYJHrXs02brXmF2+ZIGbZHmoVLUk7pPm615WCg7W9BG02AKaw4K4kbN0EjX7bx8hJdj1vIq3dDg6XRKkvDm65NmoLBC2MnoS7hkxIYpvmLOJpVREyUumYxcCnUknoc2TCMbmbOpZbi3FdMDPcPOcdo2t7jJjcxZQWU0wbCBadmfl+AlSTdQGJ4ROquqLMoDEvsp/SaQYPpju5+FV3HO6ionw4X4EopUU5zPbaIoa6akvGLOKivhc9OwnwizhtqhOMC2gXLiETqrrgTzHUOgqWiQdz2AUGyQMM0jck5f6aYU1zzUZ80nn/e4SD9DWpoGGaBQYIwiXjX9PaJQVS4Qld4EZgHZ2xLaVZ04YM1gr2JDZaeROq21sPtCA1WLKpu1NdxeoQIbrBQbsbPKoIimXqkusfVuxT0BBU1lm2LY95Eq09QkeUWtQ6vS1koSWKMNFf10pM6KQ0X4KsUZ8VGnwuQ1zzVS3WKkzuqDwvPULnZJxc5GtLXCmCj5hoqb7kidVQjcRSBCIxVzQua2VjBwIV5TBRORkTorEQqPKWPpKCxSyu0QF5RR2pxh7HUaqTFtitSqqbykYroEp9SoWaw5W6OqTXB/H9H3cGl7iZXmnRhBB1QQKgsaC1epSor5ARbwsanNJkZevJ4jmvdiVKeuuHC98mAissLYVl6obgLEvNIHIE8CMYiAE3oFGu7QgY2sQGVvE7QsbSOjh4ir57MBpPeYW8aWVeE1JNPcuzVirbcyu8FA27NvmIViYE67sUQK3z4wI6mfmdnzJs4M/RuYPN+9vazixGWIcnSoehsZjFCLMbmzkgt1q7OmS2aG+9WORAXRBomZwoCc1VtieBIdmWiuSt6hpNs4HS1tgM7KLa5BqIwRKkGJn6GSN5FidJIH6KzaJhyTKCMUB1c4n6HY8A20NFf/CvVpKOV1pJCCvstgemyyGxjGDcxZqU3YwPpi6YIJQ0bpmYmRcGVGZabKxDdI/LpZNwVGM6iNApdLUjtEtYOpVLdQau6x4Vtqcb5q4I1RniDAVyo0HhMvqiYNawXdrGyobB5Xjb0JRakjFFaJ6k2t3Ah8Q6ENVPCoV42+CXpeNf6C6gUvFaOFahS8zS9yc930u2ogDUyMUqKKfanv4JJtT5KK2RXzb+RS68RmgBq6aN9mSAN+bKH4R1fMwJFKDKXqy8MLqOKtXkhbC0g/1/uE90lKaaYIL1T9ukkRis5TV3f4Xq2jKSmZNbc1bG9unmGk1heRV02MqkU4984XKhq3UGux0EGVdotZYTFdvW50ZBzdOVxgYbGjYyGLzBsqRgU8UGfLRmA88wqKRlN61hnGs2ygaHllgM6WjWCPympfcYBN+prgjkNbatY8UGe7Kl6zd9m/gMIy5POa9AnSCMV/CRh3dbarMq7/KVbU+j7lHH7pt8QVNWKk+jSVUX0jFS8jtUNNU9mGisccoTE9oKujsRXUEGx/ELyIoPzhf19jjKOSIgAA"}')
for name, encoded in packed.items():
    destination = ROOT / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(gzip.decompress(base64.b64decode(encoded)))
print(f"Materialized {len(packed)} files ({sum((ROOT / name).stat().st_size for name in packed):,} bytes)")


In [ ]:
import subprocess, sys, time
root = "/kaggle/working/asc_tcn_t4_suite"
command = [
    sys.executable,
    f"{root}/run_tcn_t4_latency.py",
    "--input-dir", f"{root}/inputs",
    "--output-dir", f"{root}/outputs",
    "--archive", "/kaggle/working/asc_tcn_t4_latency_results.zip",
    "--latency-repeats", str(LATENCY_REPEATS),
]
if SMOKE:
    command.append("--smoke")
started = time.time()
subprocess.run(command, check=True)
print("Elapsed minutes:", (time.time() - started) / 60)


In [ ]:
from pathlib import Path
import json, pandas as pd
from IPython.display import FileLink, display

output = Path("/kaggle/working/asc_tcn_t4_suite/outputs")
seed_metrics = pd.read_csv(output / "tcn_t4_seed_metrics.csv")
summary = pd.read_csv(output / "tcn_t4_summary.csv", index_col=0)
display(seed_metrics[[
    "seed", "macro_MAPE", "Q_MAPE", "Re_MAPE", "parameters",
    "train_seconds", "inference_ms_per_trajectory",
    "inference_ms_per_trajectory_repeat_std", "n_eval_rows", "n_eval_cells", "epochs",
]])
display(summary)
print(json.dumps(json.loads((output / "tcn_t4_environment_manifest.json").read_text()), indent=2))
archive = Path("/kaggle/working/asc_tcn_t4_latency_results.zip")
print("DOWNLOAD THIS FILE:", archive, f"({archive.stat().st_size / 1e6:.2f} MB)")
display(FileLink(str(archive)))
